In [0]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print(torch.cuda.is_available())
print(torch.cuda.get_arch_list())

## Cargar Librerias y Path de datos

In [0]:
# Librerias
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer 
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

from xgboost import XGBRegressor


import shap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
DATA_PATH = Path.home() / "Downloads" / "Proyecto Desarrollo de Soluciones" / "KMC-400-20y-libro  45-DatosCompletos.xlsx"
OUTPUT_DIR = Path.home() / "Downloads" / "Proyecto Desarrollo de Soluciones" / "modelo2_outputs"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

TARGET_COL = "WASI_FullScale4compositescore"
MAX_MISSING = 0.80
MIN_NUMERIC_RATIO = 0.70
USE_MICE = False


## Carga de datos

In [0]:
raw = pd.read_excel(DATA_PATH, sheet_name="Sheet1", engine="openpyxl")
print("Forma original:", raw.shape)
raw.head()

## Limpieza BASICA de datos
Preguntar a Jessica de los datos tratados

In [0]:
ERROR_TOKENS = ["#NULL!", "#DIV/0!", "#VALUE!", "#REF!", "#NAME?", "#N/A"]

def clean_excel_tokens(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.replace(ERROR_TOKENS, np.nan)
    out = out.replace(r"^\s*$", np.nan, regex=True)
    return out

data = clean_excel_tokens(raw)
print("Celdas con #NULL! después de limpiar:", int((data == "#NULL!").sum(numeric_only=False).sum()))

## Definicion de modulos
Se definieron 8 modulos acorde a la propuesta inicial del proyecto en el ciclo pasado

In [0]:
MODULE_PREFIXES = {
    "socioeconomico": ["SCB_", "EDU_", "PROF_", "ING_", "EC_", "ICFES_"],
    "nacimiento": ["BIRTH_", "PRE_", "birth"],
    "neonatal": ["NEO_", "HOSP_", "VENT_", "OXIG", "SEPSIS", "PEDIATRIC_", "AUDIO_", "NEURO_"],
    "seguimiento_0_12m": ["FOLL12M_", "NUT12M_", "FOLLOW_", "EX41_", "HOME12_"],
    "desarrollo_psicomotor": ["PMD_", "DQ_", "NEUROSENS_", "APEGO_", "CONNERS_"],
    "home_12m": ["HOMET_", "HOME_", "DOMICILIARYVIST_", "STRANGESITUATION_"],
    "outcome_20a": ["WASI_", "TAP_", "CVLT_", "VMI_", "METP_", "ABCLP_", "ABCLBF_", "ASR_", "RT_", "NHPT_", "SURV24_", "SURVEY24YEARS_", "Years15_", "Kidscreen", "CONNERS_"],
    "neuroimagen": ["ASEG_", "aseg_", "Aseg_", "TRAC_", "APARC_", "BROADMAN_", "GMPARC_", "CAM_", "ROI_"],
}

ID_COLS = {"code", "BirthDate", "Exam15years", "filter_$", "socialcharacteristicsatbirth"}

LEAK_PREFIXES = ["WASI_", "TAP_", "CVLT_", "VMI_", "METP_", "ABCLP_", "ABCLBF_", "ASR_", "RT_", "NHPT_", "SURV24_", "SURVEY24YEARS_", "Years15_", "Kidscreen"]

def starts_with_any(col: str, prefixes: list[str]) -> bool:
    return any(str(col).startswith(p) for p in prefixes)

def columns_for_module(df: pd.DataFrame, module_name: str) -> list[str]:
    prefixes = MODULE_PREFIXES[module_name]
    return [c for c in df.columns if starts_with_any(c, prefixes)]

module_cols = {m: columns_for_module(data, m) for m in MODULE_PREFIXES}
for m, cols in module_cols.items():
    print(f"{m:>22}: {len(cols):4d} columnas")


## Missingness

In [0]:
def summarize_missingness(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    if not cols:
        return pd.Series(dtype=float)
    return df[cols].isna().mean().sort_values(ascending=False)

missing_summary = pd.DataFrame({
    m: summarize_missingness(data, cols).mean() if cols else np.nan
    for m, cols in module_cols.items()
}, index=["missing_promedio"]).T.sort_values("missing_promedio", ascending=False)

missing_summary

## Cohorte de analisis (nacimiento prevarturo -> Target)

In [0]:
analysis_df = data.loc[data["preterm"].eq(1)].copy()
analysis_df[TARGET_COL] = pd.to_numeric(analysis_df[TARGET_COL], errors="coerce")
analysis_df = analysis_df.loc[analysis_df[TARGET_COL].notna()].copy()

print("Cohorte prematuros con target observado:", analysis_df.shape)
display(analysis_df[["preterm", "ubicac", "lessthan1801g", TARGET_COL]].head())
print("\nDistribución de 'ubicac' dentro de la cohorte:")
print(analysis_df["ubicac"].value_counts(dropna=False))


## Seleccion de predictores

In [0]:
CLINICAL_MODULES = ["socioeconomico", "nacimiento", "neonatal", "seguimiento_0_12m", "desarrollo_psicomotor", "home_12m"]

def build_feature_lists(df: pd.DataFrame):
    clinical_cols = []
    neuro_cols = []
    for col in df.columns:
        if col == TARGET_COL or col in ID_COLS:
            continue
        if starts_with_any(col, LEAK_PREFIXES):
            continue
        if starts_with_any(col, MODULE_PREFIXES["neuroimagen"]):
            neuro_cols.append(col)
        elif any(starts_with_any(col, MODULE_PREFIXES[m]) for m in CLINICAL_MODULES):
            clinical_cols.append(col)
        elif col in {"ubicac", "lessthan1801g", "preterm"}:
            clinical_cols.append(col)
    return clinical_cols, neuro_cols

clinical_cols_raw, neuro_cols_raw = build_feature_lists(analysis_df)
print("Predictores clínicos brutos:", len(clinical_cols_raw))
print("Predictores neuroimagen brutos:", len(neuro_cols_raw))

def drop_high_missing(df: pd.DataFrame, cols: list[str], max_missing: float = MAX_MISSING) -> list[str]:
    if not cols:
        return []
    miss = df[cols].isna().mean()
    return [c for c in cols if miss[c] <= max_missing]

clinical_cols = drop_high_missing(analysis_df, clinical_cols_raw, MAX_MISSING)
neuro_cols = drop_high_missing(analysis_df, neuro_cols_raw, MAX_MISSING)

print("Predictores clínicos tras filtro de missing:", len(clinical_cols))
print("Predictores neuroimagen tras filtro de missing:", len(neuro_cols))


## Inferencia
Inferir tipos y columnas que no son relevantes

In [0]:
def normalize_feature_types(df_features: pd.DataFrame, min_numeric_ratio: float = MIN_NUMERIC_RATIO, cat_max_unique: int = 20):
    X = df_features.copy()
    numeric_cols, categorical_cols, dropped_cols = [], [], []

    for col in list(X.columns):
        s = X[col]

        if pd.api.types.is_numeric_dtype(s):
            numeric_cols.append(col)
            continue

        numeric_candidate = pd.to_numeric(s, errors="coerce")
        ratio_numeric = numeric_candidate.notna().mean()

        if ratio_numeric >= min_numeric_ratio:
            X[col] = numeric_candidate
            numeric_cols.append(col)
        else:
            nunique = s.nunique(dropna=True)
            if 2 <= nunique <= cat_max_unique:
                X[col] = s.astype("string")
                categorical_cols.append(col)
            else:
                dropped_cols.append(col)

    constant_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
    if constant_cols:
        X = X.drop(columns=constant_cols)
        numeric_cols = [c for c in numeric_cols if c not in constant_cols]
        categorical_cols = [c for c in categorical_cols if c not in constant_cols]
        dropped_cols.extend(constant_cols)

    return X, numeric_cols, categorical_cols, sorted(set(dropped_cols))

def make_design_matrix(df: pd.DataFrame, cols: list[str]):
    X = df[cols].copy()
    X, numeric_cols, categorical_cols, dropped_cols = normalize_feature_types(X)
    return X, numeric_cols, categorical_cols, dropped_cols

X_clinical, num_clinical, cat_clinical, dropped_clinical = make_design_matrix(analysis_df, clinical_cols)
X_multimodal_raw = analysis_df[clinical_cols + neuro_cols].copy()
X_multimodal, num_multi, cat_multi, dropped_multi = normalize_feature_types(X_multimodal_raw)

y = analysis_df[TARGET_COL].astype(float)

print("Forma clínica:", X_clinical.shape, "| numéricas:", len(num_clinical), "| categóricas:", len(cat_clinical))
print("Forma multimodal:", X_multimodal.shape, "| numéricas:", len(num_multi), "| categóricas:", len(cat_multi))
print("Columnas descartadas en clínica:", len(dropped_clinical))
print("Columnas descartadas en multimodal:", len(dropped_multi))

## Submuestra

In [0]:
neuro_available_mask = analysis_df[neuro_cols].notna().any(axis=1)
X_clinical_sub = X_clinical.loc[neuro_available_mask].reset_index(drop=True)
X_multimodal_sub = X_multimodal.loc[neuro_available_mask].reset_index(drop=True)
y_sub = y.loc[neuro_available_mask].reset_index(drop=True)

print("Cohorte clínica para comparación:", X_clinical.shape[0], "filas")
print("Submuestra con neuroimagen disponible:", X_clinical_sub.shape[0], "filas")

## Pipeline con el modelo

In [0]:
def sanitize_frame(X):
    return pd.DataFrame(X).replace({pd.NA: np.nan, None: np.nan})


def to_numeric_df(X):
    X = sanitize_frame(X)
    return X.apply(pd.to_numeric, errors="coerce")


def to_object_df(X):
    X = sanitize_frame(X)
    return X.astype(str).replace("nan", np.nan)


def build_preprocessor(numeric_cols, categorical_cols):

    numeric_pipeline = Pipeline([
        ("to_numeric", FunctionTransformer(
            to_numeric_df,
            feature_names_out="one-to-one"
        )),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("to_object", FunctionTransformer(
            to_object_df,
            feature_names_out="one-to-one"
        )),
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )),
    ])

    transformers = []

    if numeric_cols:
        transformers.append(("num", numeric_pipeline, numeric_cols))

    if categorical_cols:
        transformers.append(("cat", categorical_pipeline, categorical_cols))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True
    )


def build_model():
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def evaluate_dataset(
    X: pd.DataFrame,
    y: pd.Series,
    numeric_cols,
    categorical_cols,
    label: str = "modelo"
):
    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    model = build_model()

    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model),
    ])

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_pred = cross_val_predict(pipe, X, y, cv=cv, n_jobs=1)

    metrics = {
        "modelo": label,
        "n_filas": int(X.shape[0]),
        "n_variables": int(X.shape[1]),
        "MAE": float(mean_absolute_error(y, oof_pred)),
        "RMSE": float(mean_squared_error(y, oof_pred) ** 0.5),
        "R2": float(r2_score(y, oof_pred)),
    }

    pipe.fit(X, y)

    return pipe, pd.Series(oof_pred, index=y.index), metrics

X_clinical = X_clinical.replace({pd.NA: np.nan})
X_multimodal_sub = X_multimodal_sub.replace({pd.NA: np.nan})

y = pd.to_numeric(y, errors="coerce")
y_sub = pd.to_numeric(y_sub, errors="coerce")


pipe_clinical, pred_clinical, metrics_clinical = evaluate_dataset(
    X_clinical.reset_index(drop=True),
    y.reset_index(drop=True),
    num_clinical,
    cat_clinical,
    label="clínico"
)

pipe_multi, pred_multi, metrics_multi = evaluate_dataset(
    X_multimodal_sub.reset_index(drop=True),
    y_sub.reset_index(drop=True),
    num_multi,
    cat_multi,
    label="multimodal"
)

results = pd.DataFrame([metrics_clinical, metrics_multi])
results

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metric_names = ["MAE", "RMSE", "R2"]

for ax, metric in zip(axes, metric_names):
    sns.barplot(data=results, x="modelo", y=metric, ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel(metric)

plt.tight_layout()
plt.show()

In [0]:
best_row = results.sort_values("R2", ascending=False).iloc[0]
best_label = best_row["modelo"]
best_pipe = pipe_multi if best_label == "multimodal" else pipe_clinical
best_X = X_multimodal_sub.reset_index(drop=True) if best_label == "multimodal" else X_clinical.reset_index(drop=True)

preprocess = best_pipe.named_steps["preprocess"]
model = best_pipe.named_steps["model"]

X_trans = preprocess.transform(best_X)
feature_names = preprocess.get_feature_names_out()

sample_size = min(120, X_trans.shape[0])
sample_idx = np.random.RandomState(RANDOM_STATE).choice(X_trans.shape[0], size=sample_size, replace=False)
X_shap = X_trans[sample_idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False, max_display=20)
plt.tight_layout()
plt.show()


In [0]:
best_row = results.sort_values("R2", ascending=False).iloc[0]
best_label = best_row["modelo"]

if best_label == "multimodal":
    best_pipe = pipe_multi
    best_num = num_multi
    best_cat = cat_multi
else:
    best_pipe = pipe_clinical
    best_num = num_clinical
    best_cat = cat_clinical


# guardar
joblib.dump(pipe_clinical, OUTPUT_DIR / "modelo2_clinico_v1.joblib")
joblib.dump(pipe_multi, OUTPUT_DIR / "modelo2_multimodal_v1.joblib")

joblib.dump(
    {
        "model": best_pipe,
        "numeric_cols": best_num,
        "categorical_cols": best_cat,
        "metrics": results,
    },
    OUTPUT_DIR / "modelo2_best_bundle.joblib"
)

In [0]:
# =========================================================
# EJEMPLO DE USO INFERENCIA
# =========================================================

def predict_new_data(df, bundle_path, y_true=None):

    bundle = joblib.load(bundle_path)
    pipe = bundle["model"]

    df_clean = df.replace({pd.NA: np.nan, None: np.nan})

    preds = pipe.predict(df_clean)

    results = pd.DataFrame({
        "prediccion_modelo": preds
    }, index=df.index)

    if y_true is not None:
        results["valor_real"] = y_true
        results["error"] = results["prediccion_modelo"] - results["valor_real"]
        results["abs_error"] = results["error"].abs()

    return results

bundle = OUTPUT_DIR / "modelo2_best_bundle.joblib"

preds = predict_new_data(
    X_clinical.iloc[:5],
    bundle,
    y.iloc[:5]
)

preds